# Machine Learning Zoomcamp 2026 - Homework 3
## Module: Machine Learning for Classification

In this homework, we build a lead scoring classification model to predict whether a prospective student will convert (`converted`).

### Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

DATA_URL = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df_raw = pd.read_csv(DATA_URL)
df_raw.head()

### Data Preparation
* Check if missing values are present.
* For categorical features, replace missing values with `'NA'`.
* For numerical features, replace missing values with `0.0`.

In [ ]:
cat_cols = ['lead_source', 'industry', 'employment_status', 'location']
num_cols = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
all_features = cat_cols + num_cols
target = 'converted'

print('Missing values before imputation:')
print(df_raw[all_features].isnull().sum())

df = df_raw.copy()
for col in cat_cols:
    df[col] = df[col].fillna('NA')
for col in num_cols:
    df[col] = df[col].fillna(0.0)

df.head()

### Question 1: Mode for industry
What is the most frequent observation (mode) for the column `industry`?

In [ ]:
print(df['industry'].value_counts())
q1_ans = df['industry'].mode()[0]
print(f'\nAnswer Q1: {q1_ans}')

### Question 2: Biggest correlation
Create the correlation matrix for the numerical features of your dataset.
What are the two features that have the biggest correlation?

In [ ]:
corr = df[num_cols].corr()
print(corr.round(4))

pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

corr_dict = {}
for f1, f2 in pairs:
    corr_val = corr.loc[f1, f2]
    corr_dict[f'{f1} and {f2}'] = corr_val
    print(f'{f1} and {f2}: {corr_val:.4f}')

q2_ans = max(corr_dict, key=corr_dict.get)
print(f'\nAnswer Q2: {q2_ans}')

### Split the data
* Split data into train/val/test with 60%/20%/20% distribution using `random_state=42`.
* Ensure target value `converted` is separated.

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

del df_train[target]
del df_val[target]
del df_test[target]

print(f'Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}')

### Question 3: Biggest Mutual Information (MI)
Calculate the mutual information score between `converted` and other categorical variables in the training set.
Which of these variables has the biggest mutual information score?

In [ ]:
mi_scores = {}
for col in cat_cols:
    score = mutual_info_score(df_train[col], y_train)
    rounded = round(score, 2)
    mi_scores[col] = rounded
    print(f'{col}: raw = {score:.4f}, rounded = {rounded}')

q3_ans = max(mi_scores, key=mi_scores.get)
print(f'\nAnswer Q3: {q3_ans}')

### Question 4: Accuracy
Train a logistic regression with one-hot encoding (`DictVectorizer`).
Fit model with `LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)`.
Calculate validation accuracy rounded to 2 decimal digits.

In [ ]:
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train[all_features].to_dict(orient='records'))
X_val = dv.transform(df_val[all_features].to_dict(orient='records'))

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
orig_acc = accuracy_score(y_val, y_pred)
q4_ans = round(orig_acc, 2)
print(f'Validation Accuracy: {orig_acc:.4f}, rounded: {q4_ans}')
print(f'\nAnswer Q4: {q4_ans}')

### Question 5: Feature Selection
Find the least useful feature using feature elimination.
Which of following feature has the smallest difference (`orig_acc - acc_without`)?

In [ ]:
candidates = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
diffs = {}
for feature in candidates:
    subset = [f for f in all_features if f != feature]
    dv_sub = DictVectorizer(sparse=False)
    X_tr_sub = dv_sub.fit_transform(df_train[subset].to_dict(orient='records'))
    X_va_sub = dv_sub.transform(df_val[subset].to_dict(orient='records'))
    
    m_sub = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    m_sub.fit(X_tr_sub, y_train)
    acc_sub = accuracy_score(y_val, m_sub.predict(X_va_sub))
    diff = orig_acc - acc_sub
    diffs[feature] = diff
    print(f'Without {feature}: acc={acc_sub:.4f}, diff={diff:+.6f}')

q5_ans = min(diffs, key=diffs.get)
print(f'\nAnswer Q5: {q5_ans}')

### Question 6: Parameter Tuning
Train regularized logistic regression with `C` in `[0.000001, 0.00001, 0.0001, 0.001]`.
Which `C` leads to the best accuracy on validation dataset (rounded to 3 decimal digits)?

In [ ]:
c_candidates = [0.000001, 0.00001, 0.0001, 0.001]
c_scores = {}
for c in c_candidates:
    m_c = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    m_c.fit(X_train, y_train)
    acc_c = accuracy_score(y_val, m_c.predict(X_val))
    rounded_c = round(acc_c, 3)
    c_scores[c] = rounded_c
    print(f'C={c:<8}: acc={acc_c:.5f}, rounded={rounded_c}')

best_acc = max(c_scores.values())
best_c = min([c for c, score in c_scores.items() if score == best_acc])
q6_ans = best_c
print(f'\nAnswer Q6: {q6_ans}')

### Summary of Answers
| # | Question | Answer |
|---|---|---|
| 1 | Mode for industry | **technology** |
| 2 | Biggest correlation | **interaction_count and lead_score** |
| 3 | Biggest MI | **lead_source** |
| 4 | Accuracy | **0.65** |
| 5 | Feature selection | **number_of_courses_viewed** |
| 6 | Parameter tuning | **0.001** |